Structure DC - PV Reask
=
This notebook generates AutoCAD scripts for the DC side of PV Reask. It draws the string cables (positive and negative), the DC trenches (valas) and the DC ducts (eletrodutos), routed from the strings on each structure line to their inverter.

**Workflow overview:**
1. Convert the DWG to DXF and load its entities into the `cad_entities` table of `pv_projects.duckdb` as project `"PV Reask"`.
2. Select the string blocks and the inverter blocks.
3. Decide whether each inverter is east or west of its strings.
4. Classify each string by structure type and position (`S`, `U_R` / `U_L`, `T` / `M` / `B`) and compute the structure line (`Line`) it sits on.
5. Split each structure line into `Group`s (continuous structure runs) and `SubGroup`s (the part of a run that goes to one inverter). Compute the structure ends and the string count of each SubGroup.
6. Map each string count to the ducts needed (max 3 strings per duct).
7. Classify each structure run and each string by its position relative to the inverter (8 cases), which picks one of 3 routing patterns.
8. Write `Script Cabos - Reask.scr`, `Script Valas - Reask.scr` and `Script Dutos - Reask.scr`, plus Excel files on the Desktop for checking.

**DWG prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** `PV Reask - Strings.dwg` sits in `dwg_folder` (`422 - Zagope - UFV Reask e Tooletown\1 - Reask Solar Farm\...\01 - Layout`). The DXF is written to the same folder and deleted at the end. The ODA File Converter 27.1.0 must be installed at the path set in `DWG_to_DuckDB_ETL.py`. The converter converts every DWG in `temp_folder`, so keep only this drawing in it.
- **Database:** `load_to_duckdb` deletes every existing `"PV Reask"` row in `cad_entities` before loading (the same project name is used by `BOQ - Reask`, so each load replaces the other's data). `pv_projects.duckdb` is opened relative to the working directory.
- **Model space only:** strings and inverters are block references (`INSERT`) placed directly in model space.
- **Block names (case-sensitive):**
  - Strings: name `LIKE 'EMF422-1_string_%'` (in `LIKE`, `_` also matches any single character).
  - Inverters: exactly `EMF422-1_inversor`.
  - 2-string structure: the classification step recognises it only by the exact name `EMF422-1_string_2`. It has two blocks at the same point: rotation `0` = right half (`U_R`), any other rotation = left half (`U_L`).
  - Any other string block: blocks stacked at the same X (±0.5) and less than 5 apart in Y are `T` / `M` / `B`, and a block with no stacked neighbour is `S`.
- **Attributes:**
  - Inverter blocks carry `UFV`, `PST` and `INV`. The code rebuilds the inverter key as `UFV.PST.INV`, with `UFV` zero-padded to 2 digits and `PST` / `INV` used as they are (so they must already be 2-digit, e.g. `01`), and the inverter's own `GRP` attribute is ignored.
  - Every string block carries `GRP` = its inverter key in exactly the same text format (e.g. `01.01.01`). Strings whose `GRP` doesn't match an inverter are dropped from the scripts.
- **Layout and geometry** (hardcoded values, tuned to the Reask structures):
  - Structure lines run east-west.
  - Stacked strings are about 2.32 apart (structure line = Y ∓ 2.3204 for `T` / `B`).
  - Structure ends are 5.193 (`S`), 11.053 (`U_R` / `U_L`) or 16.913 (`T` / `M` / `B`) from the block insertion point.
  - An X gap of more than 40 between consecutive strings on a line means a new structure run.
  - The inverters sit between structure lines: 6.6553 below the line above them (`distance1`) and 2 above the line below them (`distance2`).
- **When running the `.scr` in AutoCAD:**
  - The layers `EMF_cabo_solar_positivo`, `EMF_cabo_solar_negativo`, `EMF_DC_vala`, `EMF_bt_eletroduto_1S`, `EMF_bt_eletroduto_2S` and `EMF_bt_eletroduto_3S` must already exist (`LAYER S` doesn't create them).
  - The scripts set `PLINEWID` (0 for cables, 0.5 for trenches, 0.03 for ducts), and the value stays set after the script ends.
  - Turn off running object snaps (`OSMODE = 0`) so the script's points aren't snapped to nearby geometry.

Extracting the AutoCAD data and loading it into the duckdb database.
=

- Copies `PV Reask - Strings.dwg` into `temp_folder`. `convert_dwg_to_dxf` runs the ODA File Converter on that folder and writes the DXF (ACAD2018) into `output_folder` (= `dwg_folder`).
- `load_to_duckdb` reads every model space entity with `ezdxf` and loads it into `cad_entities` as `"PV Reask"`. It deletes the project's old rows first. For blocks it stores the name, insertion point (`X`, `Y`), rotation, color and the attribute tags (`UFV`, `PST`, `INV`, `STR`, `GRP`, ...).
- The DXF and the temporary DWG are deleted. The connection `conn` stays open and is closed at the end of the script-writing step.

In [1]:
from DWG_to_DuckDB_ETL import *
import os
import shutil
import numpy as np

dwg_folder = r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\422 - Zagope - UFV Reask e Tooletown\1 - Reask Solar Farm\01 - Projetos\01 - Em Confecção\02 - Edição\01 - Layout"
temp_folder = r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\422 - Zagope - UFV Reask e Tooletown\1 - Reask Solar Farm\01 - Projetos\01 - Em Confecção\02 - Edição\01 - Layout\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Reask - Strings.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Reask - Strings.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Reask - Strings.dwg")

conn = load_to_duckdb(dxf_file, "PV Reask")
print("Data Loaded")
print("DuckDB Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Project PV Reask already exists
Removed existing data for project: PV Reask
Successfully loaded project
Loaded 19289 entities for project: PV Reask
Don't forget to close connection.
Data Loaded
DuckDB Loaded


Loading the data from PV Reask
=

- `data_ELM` = relation with `layer`, `name`, `X`, `Y`, `UFV`, `PST`, `INV`, `GRP`, `STR` and `rotation` for `PV Reask`. The next cells query it by its Python variable name.
- Layer count: it runs on the whole `cad_entities` table (no project filter), so it counts every project in the database. Also, `conn.execute(...)` returns the connection, so `print(result)` doesn't print the table (`conn.sql(...).show()` would). It is for information only.

In [2]:
data_ELM = conn.sql("SELECT layer, name, X, Y, UFV, PST, INV, GRP, STR, rotation FROM cad_entities WHERE project_name = 'PV Reask'")

print(data_ELM)

result = conn.execute("""
    SELECT
        layer,
        COUNT(*) as count
    FROM cad_entities
    GROUP BY layer
    ORDER BY count
""")

print(result)

┌─────────────────────────┬───────────────────────┬───────────┬───────────┬─────────┬─────────┬─────────┬─────────┬─────────┬──────────┐
│          layer          │         name          │     X     │     Y     │   UFV   │   PST   │   INV   │   GRP   │   STR   │ rotation │
│         varchar         │        varchar        │   float   │   float   │ varchar │ varchar │ varchar │ varchar │ varchar │  float   │
├─────────────────────────┼───────────────────────┼───────────┼───────────┼─────────┼─────────┼─────────┼─────────┼─────────┼──────────┤
│ EMF_em_pst_1.1          │ EMF422-1_eletrocentro │ 700595.75 │  756504.5 │         │         │ NULL    │ NULL    │ NULL    │      0.0 │
│ EMF_em_pst_1.3          │ EMF422-1_eletrocentro │  700624.3 │ 756297.44 │         │         │ NULL    │ NULL    │ NULL    │    180.0 │
│ EMF_em_pst_1.2          │ EMF422-1_eletrocentro │  700364.6 │ 756310.75 │         │         │ NULL    │ NULL    │ NULL    │      0.0 │
│ EMF_em_pst_1.4          │ EMF422-1_elet

Separating the data into different dataframes.
=
First one is for strings.

- Selects the string blocks (`name LIKE 'EMF422-1_string_%'`) from `data_ELM` into `strings_df` (5488 strings in the saved output) and prints a summary.
- One row per string, with `X`, `Y`, `rotation` and the `UFV`, `PST`, `INV`, `GRP`, `STR` attributes.

In [3]:
strings = conn.sql("SELECT * FROM data_ELM WHERE name LIKE 'EMF422-1_string_%'")

strings_df = strings.fetchdf()
print(strings)
print(strings_df.describe())
print(strings_df.info())

┌────────────────┬───────────────────┬───────────┬───────────┬─────────┬─────────┬─────────┬──────────┬─────────┬──────────┐
│     layer      │       name        │     X     │     Y     │   UFV   │   PST   │   INV   │   GRP    │   STR   │ rotation │
│    varchar     │      varchar      │   float   │   float   │ varchar │ varchar │ varchar │ varchar  │ varchar │  float   │
├────────────────┼───────────────────┼───────────┼───────────┼─────────┼─────────┼─────────┼──────────┼─────────┼──────────┤
│ EMF_em_pst_1.1 │ EMF422-1_string_2 │ 700354.75 │ 756686.25 │ 01      │ 01      │ 01      │ 01.01.01 │ 01      │      0.0 │
│ EMF_em_pst_1.1 │ EMF422-1_string_2 │ 700354.75 │ 756686.25 │ 01      │ 01      │ 01      │ 01.01.01 │ 02      │    180.0 │
│ EMF_em_pst_1.1 │ EMF422-1_string_3 │ 700361.25 │  756677.6 │ 01      │ 01      │ 01      │ 01.01.01 │ 03      │      0.0 │
│ EMF_em_pst_1.1 │ EMF422-1_string_3 │ 700361.25 │ 756675.25 │ 01      │ 01      │ 01      │ 01.01.01 │ 04      │      0.0 │


Second one is for the inverters.

- Selects the inverter blocks by exact name `EMF422-1_inversor`.
- `UFV` is zero-padded to 2 digits, and `GRP` is rebuilt as `UFV.PST.INV` (the inverter's own `GRP` attribute is empty in this drawing). `PST` and `INV` are used as they are, so they must already be 2-digit to match the strings' `GRP` (e.g. `01.01.01`).
- Renames `X` / `Y` to `inverter_x` / `inverter_y`.

In [4]:
inverter = conn.sql("SELECT layer, name, X, Y, UFV, PST, INV, GRP FROM data_ELM WHERE name = 'EMF422-1_inversor'")
print(inverter)

inverter_df = inverter.fetchdf()
inverter_df["UFV"] = inverter_df["UFV"].astype(str).str.zfill(2)
inverter_df['GRP'] = inverter_df["UFV"].astype(str) + "." + inverter_df["PST"].astype(str) + "." + inverter_df["INV"].astype(str)
inverter_df.rename(columns={'X': 'inverter_x', 'Y': 'inverter_y'}, inplace=True)

print(inverter_df.describe())
print(inverter_df.info())
print(inverter_df)

┌────────────────┬───────────────────┬───────────┬───────────┬─────────┬─────────┬─────────┬─────────┐
│     layer      │       name        │     X     │     Y     │   UFV   │   PST   │   INV   │   GRP   │
│    varchar     │      varchar      │   float   │   float   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼───────────────────┼───────────┼───────────┼─────────┼─────────┼─────────┼─────────┤
│ EMF_em_pst_1.1 │ EMF422-1_inversor │ 700436.25 │ 756667.25 │ 01      │ 01      │ 01      │         │
│ EMF_em_pst_1.1 │ EMF422-1_inversor │ 700520.25 │ 756634.25 │ 01      │ 01      │ 02      │         │
│ EMF_em_pst_1.1 │ EMF422-1_inversor │ 700595.25 │ 756601.25 │ 01      │ 01      │ 04      │         │
│ EMF_em_pst_1.1 │ EMF422-1_inversor │ 700597.94 │ 756568.25 │ 01      │ 01      │ 06      │         │
│ EMF_em_pst_1.1 │ EMF422-1_inversor │ 700597.94 │ 756535.25 │ 01      │ 01      │ 09      │         │
│ EMF_em_pst_1.1 │ EMF422-1_inversor │ 700597.94 │ 756524.25 │ 01      │ 

Comparing the coordinates of the inverter to the mean of the X coordinate from all the strings that share a GRP with the inverter, in order to determine whether is the inverter is to the west (W) or east (E) in relation to their strings.
=

- `Comparing X` = the mean X of all strings with the same `GRP`, i.e. the centre of the inverter's string field.
- Inner merge with `inverter_df` on `GRP`. The two printed lengths (`1:` / `2:`) show whether strings were lost: here 5488 → 5461, so 27 strings have a `GRP` with no matching inverter. Those strings are also dropped from the cable script, and their SubGroups from the trench and duct scripts.
- `INVERTER - E or W` = `E` if `inverter_x` ≥ `Comparing X`, otherwise `W`.
- `inverter_position` keeps one row per `GRP`. It is used later by the relative position function.

In [5]:
struct_df = strings_df.copy()
struct_df['Comparing X'] = struct_df.groupby('GRP')['X'].transform('mean')
print(f'1:{len(struct_df)}')
struct_df = struct_df.merge(inverter_df, on='GRP')
print(f'2:{len(struct_df)}')
struct_df['INVERTER - E or W'] = np.where(
    struct_df['inverter_x'] >= struct_df['Comparing X'],
    'E',
    'W'
)
inverter_position = struct_df[['GRP', 'INVERTER - E or W']].drop_duplicates("GRP").sort_values(by='GRP').reset_index(drop=True)
print(inverter_position)

1:5488
2:5461
          GRP INVERTER - E or W
0    01.01.01                 E
1    01.01.02                 E
2    01.01.03                 E
3    01.01.04                 E
4    01.01.05                 E
..        ...               ...
194  03.03.16                 E
195  03.03.17                 E
196  03.03.18                 E
197  03.03.19                 E
198  03.03.20                 E

[199 rows x 2 columns]


Iterating through the strings dataframe to determine the classification of each string.
=

S = 1-string structure.

U_R = Middle part of structure on the right on a 2-string structure.
U_L = Middle part of structure on the left on a 2-string structure.

T = Top string of a 3-string structure.
M = Middle string of a 3-string structure.
B = Bottom string of a 3-string structure.

- For each string, it looks for stacked neighbours: other strings less than 0.5 away in X and 5 in Y, with a different Y.
- `EMF422-1_string_2` is checked first: rotation `0` → `U_R`, otherwise `U_L`.
- Any other block with neighbours: above all of them → `T`, above some → `M`, below all → `B`. Without neighbours → `S`.
- `Line` = the Y of the structure centre, rounded to an integer: `S`, `U_R`, `U_L` and `M` → Y, `T` → Y − 2.3204, `B` → Y + 2.3204.
- Sorts by `Line` descending (north to south), then X ascending (west to east). The break conditions in the next step depend on this order.
- The loop runs row by row (`iterrows`) and compares against every string, so it gets slow on large plants.
- `value_counts()`: on complete 3-high structures, the `T`, `M` and `B` counts are equal, and so are the `U_R` and `U_L` counts.

In [6]:
for idx, row in strings_df.iterrows():
    x = row['X']
    y = row['Y']


    mask_TMB = (
        (strings_df['Y'] >= y - 5) &
        (strings_df['Y'] <= y + 5) &
        (strings_df['X'] >= x - 0.5) &
        (strings_df['X'] <= x + 0.5) &
        (strings_df['Y'] != y) &
        (strings_df.index != idx)
    )

    if row['name'] == 'EMF422-1_string_2':
        if row['rotation'] == 0:
            strings_df.loc[idx, 'Type'] = "U_R"
        else:
            strings_df.loc[idx, 'Type'] = "U_L"

    elif mask_TMB.any():
        y_neighbors = strings_df.loc[mask_TMB, 'Y']

        if (y > y_neighbors).all():
            strings_df.loc[idx, 'Type'] = "T"
        elif (y > y_neighbors).any():
            strings_df.loc[idx, 'Type'] = "M"
        else:
            strings_df.loc[idx, 'Type'] = "B"

    else:
        strings_df.loc[idx, 'Type'] = "S"


strings_df.loc[:,'Line'] = np.where(
    (strings_df["Type"] == "S") | (strings_df["Type"] == "U_R") | (strings_df["Type"] == "U_L") | (strings_df["Type"] == "M"),
    round(strings_df["Y"]),
    np.where(
        strings_df["Type"] == "T",
        round(strings_df["Y"] - 2.3204),
        round(strings_df["Y"] + 2.3204)
    )
)


strings_df.sort_values(by=["Line", "X"], ascending=[False,True], inplace=True, ignore_index=True)
print(strings_df)
print(strings_df['Type'].value_counts())

               layer               name          X            Y UFV PST INV  \
0     EMF_em_pst_1.1  EMF422-1_string_2  700354.75  756686.2500  01  01  01   
1     EMF_em_pst_1.1  EMF422-1_string_2  700354.75  756686.2500  01  01  01   
2     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756677.6250  01  01  01   
3     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756675.2500  01  01  01   
4     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756672.9375  01  01  01   
...              ...                ...        ...          ...  ..  ..  ..   
5483  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755228.6250  03  03  20   
5484  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755228.6250  03  03  20   
5485  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755218.1250  03  03  20   
5486  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755218.1250  03  03  20   
5487  EMF_em_pst_3.3  EMF422-1_string_1  700151.75  755207.6250  03  03  20   

           GRP STR  rotation Type      Line  
0    

Conditions to break the strings into groups and subgroups.
=

Each row is compared with the previous one (in the sort order above):
- `condition1`: the `Line` changes.
- `condition2`: the X gap to the previous string is more than 40 (the structure run is interrupted).
- `condition3`: the `GRP` (inverter key) changes. The comment says "Different PSTs", but it compares the full inverter key.

`Group` = cumulative sum of conditions 1 and 2, i.e. one continuous structure run. `SubGroup` also breaks on condition 3, so it is the part of a run that goes to one inverter.

In [7]:
#Different Lines
condition1 = strings_df["Line"] != strings_df["Line"].shift()

#Break in the Structure
condition2 = abs(strings_df["X"] - strings_df["X"].shift()) > 40

#Different PSTs
condition3 = strings_df["GRP"] != strings_df["GRP"].shift()



break_condition_1 = condition1 | condition2 | condition3
break_condition_2 = condition1 | condition2

strings_df['SubGroup'] = break_condition_1.cumsum()
strings_df['Group'] = break_condition_2.cumsum()
print(strings_df)

               layer               name          X            Y UFV PST INV  \
0     EMF_em_pst_1.1  EMF422-1_string_2  700354.75  756686.2500  01  01  01   
1     EMF_em_pst_1.1  EMF422-1_string_2  700354.75  756686.2500  01  01  01   
2     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756677.6250  01  01  01   
3     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756675.2500  01  01  01   
4     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756672.9375  01  01  01   
...              ...                ...        ...          ...  ..  ..  ..   
5483  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755228.6250  03  03  20   
5484  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755228.6250  03  03  20   
5485  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755218.1250  03  03  20   
5486  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755218.1250  03  03  20   
5487  EMF_em_pst_3.3  EMF422-1_string_1  700151.75  755207.6250  03  03  20   

           GRP STR  rotation Type      Line  SubGro

W exit means the west-most exit of the structure line.
E exit means the east-most exit of the structure line.

Also creates the Dataframe for the cable generation.

- `W exit` / `E exit` per string = X ∓ 5.193 (`S`), ∓ 11.053 (`U_R` / `U_L`, both halves share the insertion point), ∓ 16.913 (`T` / `M` / `B`). These are half the structure lengths, measured from the block insertion point.
- `SubGroup - Position X - W/E` and `Group - Position X - W/E` = the minimum `W exit` / maximum `E exit` of each SubGroup / Group.
- `cables_df` = `strings_df` inner-merged with `inverter_df` on `GRP`: one row per string, with its inverter coordinates. Strings without an inverter are dropped. The duplicated columns get `_x` / `_y` suffixes and are dropped in the cable routing step. It is built before `No. of Strings`, so it doesn't have that column.
- `No. of Strings` = the number of strings in each SubGroup. The printed `value_counts()` counts strings, not SubGroups, and its sum is the total number of strings.

In [8]:
strings_df['W exit'] = np.where(
    strings_df["Type"] == "S",
    strings_df['X'] - 5.193,
    np.where(
        (strings_df["Type"] == "U_R") | (strings_df["Type"] == "U_L"),
        strings_df['X'] - 11.053,
        strings_df['X'] - 16.913
    )
)

strings_df['E exit'] = np.where(
    strings_df["Type"] == "S",
    strings_df['X'] + 5.193,
    np.where(
        (strings_df["Type"] == "U_R") | (strings_df["Type"] == "U_L"),
        strings_df['X'] + 11.053,
        strings_df['X'] + 16.913
    )
)

strings_df['SubGroup - Position X - W'] = strings_df.groupby("SubGroup")["W exit"].transform('min')
strings_df['SubGroup - Position X - E'] = strings_df.groupby("SubGroup")["E exit"].transform('max')

strings_df['Group - Position X - W'] = strings_df.groupby("Group")["W exit"].transform('min')
strings_df['Group - Position X - E'] = strings_df.groupby("Group")["E exit"].transform('max')

cables_df = strings_df.merge(inverter_df, on="GRP")

strings_df['No. of Strings'] = strings_df.groupby("SubGroup")['name'].transform('count')

print(strings_df)
print(strings_df['No. of Strings'].value_counts())
print(strings_df['No. of Strings'].value_counts().sum())

               layer               name          X            Y UFV PST INV  \
0     EMF_em_pst_1.1  EMF422-1_string_2  700354.75  756686.2500  01  01  01   
1     EMF_em_pst_1.1  EMF422-1_string_2  700354.75  756686.2500  01  01  01   
2     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756677.6250  01  01  01   
3     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756675.2500  01  01  01   
4     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756672.9375  01  01  01   
...              ...                ...        ...          ...  ..  ..  ..   
5483  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755228.6250  03  03  20   
5484  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755228.6250  03  03  20   
5485  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755218.1250  03  03  20   
5486  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755218.1250  03  03  20   
5487  EMF_em_pst_3.3  EMF422-1_string_1  700151.75  755207.6250  03  03  20   

           GRP STR  rotation  ...      Line  SubGro

Grouping the data per Subgroup

- One row per SubGroup with `groupby("SubGroup").first()`. The positions and counts are the same for every row of a SubGroup.
- It also keeps the first string's `W exit` / `E exit`, which aren't used later.
- Renames: `Line` → `Y`, SubGroup ends → `X-W-STR` / `X-E-STR`, Group (structure run) ends → `X-W-STRUCT` / `X-E-STRUCT`.
- Prints the maximum number of strings in a SubGroup. It must be covered by `duct_map` in the next step (22 here).

In [9]:
groups_df = strings_df.groupby("SubGroup")[['GRP','Line', 'SubGroup - Position X - W', 'SubGroup - Position X - E', 'Group - Position X - W', 'Group - Position X - E','W exit', 'E exit', 'No. of Strings']].first()
groups_df.rename(columns={'Line': 'Y', 'Group - Position X - W': 'X-W-STRUCT', 'Group - Position X - E': 'X-E-STRUCT', 'SubGroup - Position X - W': 'X-W-STR', 'SubGroup - Position X - E': 'X-E-STR'}, inplace=True)
print(f'Maximum number of strings in a group: {groups_df['No. of Strings'].max()}\n')
print(groups_df)

Maximum number of strings in a group: 22

               GRP         Y      X-W-STR      X-E-STR   X-W-STRUCT  \
SubGroup                                                              
1         01.01.01  756686.0  700343.6875  700365.8125  700343.6875   
2         01.01.01  756675.0  700344.3125  700401.6250  700344.3125   
3         01.01.01  756664.0  700344.9375  700437.4375  700344.9375   
4         01.01.01  756653.0  700345.6250  700473.2500  700345.6250   
5         01.01.01  756642.0  700346.2500  700380.1250  700346.2500   
...            ...       ...          ...          ...          ...   
626       03.03.20  755250.0  700069.1875  700173.3750  700069.1875   
627       03.03.20  755239.0  700111.1875  700168.5000  700111.1875   
628       03.03.20  755229.0  700140.6875  700162.8125  700140.6875   
629       03.03.20  755218.0  700140.6875  700162.8125  700140.6875   
630       03.03.20  755208.0  700134.8125  700168.6875  700134.8125   

           X-E-STRUCT       W exit

Duct_map is needed to represent how many ducts are necessary and how many strings are there in each duct. Given that the maximum of strings in a duct is 3, the duct_map is as follows:
=

- The key is `No. of Strings`, and the value lists the strings per duct between dashes (`-3-3-1-` = three ducts with 3, 3 and 1 strings).
- `Number of Ducts` = number of dashes − 1 (for information only; the scripts don't use it).
- A count above 22 isn't in the map and becomes `NaN` (`Number of Ducts` = −1 and no duct lines). Extend the map if the maximum printed above grows.
- Merges with `inverter_df` on `GRP` (inner join). SubGroups without an inverter are dropped without warning (here 630 → 628 rows).
- `W or E` = which end of the whole structure run (`X-W-STRUCT` / `X-E-STRUCT`) is closer to the inverter, and `closest_x` = the X of that end. The trench and ducts leave the structure line there.

In [10]:
duct_map = {
    1: '-1-',
    2: '-2-',
    3: '-3-',
    4: '-3-1-',
    5: '-3-2-',
    6: '-3-3-',
    7: '-3-3-1-',
    8: '-3-3-2-',
    9: '-3-3-3-',
    10: '-3-3-3-1-',
    11: '-3-3-3-2-',
    12: '-3-3-3-3-',
    13: '-3-3-3-3-1-',
    14: '-3-3-3-3-2-',
    15: '-3-3-3-3-3-',
    16: '-3-3-3-3-3-1-',
    17: '-3-3-3-3-3-2-',
    18: '-3-3-3-3-3-3-',
    19: '-3-3-3-3-3-3-1-',
    20: '-3-3-3-3-3-3-2-',
    21: '-3-3-3-3-3-3-3-',
    22: '-3-3-3-3-3-3-3-1-',
}

groups_df.loc[:,'Duct Definition'] = groups_df['No. of Strings'].map(duct_map)
groups_df['Number of Ducts'] = groups_df['Duct Definition'].astype(str).str.count('-')-1
groups_df = groups_df.merge(inverter_df, on='GRP')
groups_df['W or E'] = np.where(
    abs(groups_df['X-W-STRUCT'] - groups_df['inverter_x']) > abs(groups_df['X-E-STRUCT'] - groups_df['inverter_x']),
    "E",
    "W"
)

groups_df['closest_x'] = np.where(
    groups_df['W or E'] == "W",
    groups_df['X-W-STRUCT'],
    groups_df['X-E-STRUCT']
)
print(groups_df)

          GRP         Y      X-W-STR      X-E-STR   X-W-STRUCT   X-E-STRUCT  \
0    01.01.01  756686.0  700343.6875  700365.8125  700343.6875  700365.8125   
1    01.01.01  756675.0  700344.3125  700401.6250  700344.3125  700401.6250   
2    01.01.01  756664.0  700344.9375  700437.4375  700344.9375  700437.4375   
3    01.01.01  756653.0  700345.6250  700473.2500  700345.6250  700473.2500   
4    01.01.01  756642.0  700346.2500  700380.1250  700346.2500  700509.0625   
..        ...       ...          ...          ...          ...          ...   
623  03.03.20  755250.0  700069.1875  700173.3750  700069.1875  700173.3750   
624  03.03.20  755239.0  700111.1875  700168.5000  700111.1875  700168.5000   
625  03.03.20  755229.0  700140.6875  700162.8125  700140.6875  700162.8125   
626  03.03.20  755218.0  700140.6875  700162.8125  700140.6875  700162.8125   
627  03.03.20  755208.0  700134.8125  700168.6875  700134.8125  700168.6875   

          W exit       E exit  No. of Strings Duct 

Cables: per-string routing data
=
- Drops the duplicated merge columns (`*_x` / `*_y`) and `SubGroup` / `Group`, and renames the position columns like in `groups_df`.
- `W or E or M`: `E` if the string's `E exit` is closer to the inverter than its `W exit`, otherwise `W`. The `M` branch (inverter between the two exits) can never be true: it needs `inverter_x` < `W exit` and > `E exit` at the same time, while `W exit` < `E exit`.
- `closest_x` = the end of the structure run closest to the inverter. This is the "pile" point where the cable leaves the structure line, the same point the trench uses.
- `Exit` = the string's own end where the cable starts (`E exit` if `E`, otherwise `W exit`).
- Sorted by `GRP` and `Line`.

In [11]:
cables_df.drop(columns=['UFV_x', 'UFV_y', 'PST_x', 'PST_y', 'INV_x', 'INV_y', 'SubGroup', 'Group', 'layer_x', 'name_x', 'layer_y', 'name_y'], inplace=True)
cables_df.rename(columns={'Group - Position X - W': 'X-W-STRUCT', 'Group - Position X - E': 'X-E-STRUCT', 'SubGroup - Position X - W': 'X-W-STR', 'SubGroup - Position X - E': 'X-E-STR'}, inplace=True)
cables_df['W or E or M'] = np.where(
    abs(cables_df['W exit'] - cables_df['inverter_x']) > abs(cables_df['E exit'] - cables_df['inverter_x']),
    "E",
    np.where(
        (cables_df['inverter_x'] < cables_df['W exit']) & (cables_df['inverter_x'] > cables_df['E exit']),
        "M",
        "W"
    )
)

# THIS FINDS THE CLOSEST X TO EXIT THE STRING.
# CALCULATING WHICH ONE IS THE CLOSEST ONE (X-W-STRUCT, X-E-STRUCT).
cables_df['closest_x'] = np.where(
    abs(cables_df['X-W-STRUCT'] - cables_df['inverter_x']) < abs(cables_df['X-E-STRUCT'] - cables_df['inverter_x']),
    cables_df['X-W-STRUCT'],
    cables_df['X-E-STRUCT']
)

cables_df['Exit'] = np.where(
    cables_df['W or E or M'] == 'E',
    cables_df['E exit'],
    cables_df['W exit']
)

cables_df.sort_values(['GRP', 'Line'], inplace = True)
print(cables_df.head())

              X            Y       GRP STR  rotation Type      Line  \
26  700363.1875  756644.6250  01.01.01  27       0.0    T  756642.0   
27  700363.1875  756642.2500  01.01.01  28       0.0    M  756642.0   
15  700362.5625  756655.6250  01.01.01  16       0.0    T  756653.0   
16  700362.5625  756653.2500  01.01.01  17       0.0    M  756653.0   
17  700362.5625  756650.9375  01.01.01  18       0.0    B  756653.0   

        W exit      E exit     X-W-STR     X-E-STR  X-W-STRUCT   X-E-STRUCT  \
26  700346.250  700380.125  700346.250  700380.125  700346.250  700509.0625   
27  700346.250  700380.125  700346.250  700380.125  700346.250  700509.0625   
15  700345.625  700379.500  700345.625  700473.250  700345.625  700473.2500   
16  700345.625  700379.500  700345.625  700473.250  700345.625  700473.2500   
17  700345.625  700379.500  700345.625  700473.250  700345.625  700473.2500   

    inverter_x  inverter_y W or E or M    closest_x        Exit  
26   700436.25   756667.25      

Position relative to the inverter
=
`position_of_structure_regarding_inverter` merges `inverter_position` (inner join on `GRP`) and puts each row in one of 8 cases, coded `R/L_A/B_E/W`:
- `R` / `L`: `closest_x` is right / left of `inverter_x` (equal counts as `L`).
- `A` / `B`: `Y` is above / below `inverter_y` (equal counts as `B`). In `cables_df`, `Y` is the string Y. In `groups_df`, it is the structure line.
- `E` / `W`: the inverter is east / west of its strings (from the E or W step).

The 8 conditions cover every case, so the `'unk'` default never shows up.

Exports `cables.xlsx` and `groups.xlsx` to the Desktop for checking (hardcoded paths). `cables.xlsx` is overwritten again in the next step.

In [12]:
# print(cables_df.columns)
# print(groups_df.columns)
def position_of_structure_regarding_inverter(df):
    """
    The function takes a Dataframe and defines where the line of structure or the string is regarding their inverter.
    Option 1: Right (R)(X coordinate) and above (A)(Y coordinate) the inverter. Inverter east of the GRP (E). Being set as "R_A_E"; or
    Option 2: Right (R)(X coordinate) and below (B)(Y coordinate) the inverter. Inverter east of the GRP (E). Being set as "R_B_E"; or
    Option 3: Left (L)(X coordinate) and above (A)(Y coordinate) the inverter. Inverter east of the GRP (E). Being set as "L_A_E"; or
    Option 4: Left (L)(X coordinate) and below (B)(Y coordinate) the inverter. Inverter east of the GRP (E). Being set as "L_B_E"; or
    Option 5: Right (R)(X coordinate) and above (A)(Y coordinate) the inverter. Inverter west of the GRP (W). Being set as "R_A_W"; or
    Option 6: Right (R)(X coordinate) and below (B)(Y coordinate) the inverter. Inverter west of the GRP (W). Being set as "R_B_W"; or
    Option 7: Left (L)(X coordinate) and above (A)(Y coordinate) the inverter. Inverter west of the GRP (W). Being set as "L_A_W"; or
    Option 8: Left (L)(X coordinate) and below (B)(Y coordinate) the inverter. Inverter west of the GRP (W). Being set as "L_B_W".
    """
    df = df.merge(inverter_position, on='GRP')

    inverter_east = df['INVERTER - E or W'] == 'E'
    inverter_west = ~inverter_east

    right_of_inverter = df['closest_x'] - df['inverter_x'] > 0
    left_of_inverter = ~right_of_inverter

    above_inverter = df['Y'] - df['inverter_y'] > 0
    below_inverter = ~above_inverter

    conditions = [
        # East cases
        inverter_east & right_of_inverter & above_inverter,
        inverter_east & right_of_inverter & below_inverter,
        inverter_east & left_of_inverter & above_inverter,
        inverter_east & left_of_inverter & below_inverter,

        # West cases
        inverter_west & right_of_inverter & above_inverter,
        inverter_west & right_of_inverter & below_inverter,
        inverter_west & left_of_inverter & above_inverter,
        inverter_west & left_of_inverter & below_inverter,
    ]

    choices = ['R_A_E', 'R_B_E', 'L_A_E', 'L_B_E', 'R_A_W', 'R_B_W', 'L_A_W', 'L_B_W']


    df['Relative_Position'] = np.select(conditions, choices, default='unk')

    return df

cables_df = position_of_structure_regarding_inverter(cables_df)
groups_df = position_of_structure_regarding_inverter(groups_df)
cables_df.to_excel(r"C:\Users\Usuario\Desktop\cables.xlsx")
groups_df.to_excel(r"C:\Users\Usuario\Desktop\groups.xlsx")

Writing the cable, trench and duct scripts
=
- `intermediary_coordinates` adds three points 1.7, 1.5 and 1.3 from the inverter, on the structure side, and moves `inverter_y` 0.1 toward the structure (in place). It runs on `cables_df` (relative to the string Y) and on `script_df` (relative to the structure line). The extra vertices make the polylines easier to stretch later.
- `make_pline(row, is_cable, offset)` builds one `_.pline` line:
  - Cables start at `(Exit, Line)` (the string end on the structure line).
  - Then `(closest_x, Line)`, the pile point at the end of the structure run.
  - With an `offset`: vertical along `closest_x` to `inverter_y + offset`, then horizontal to `inverter_x` (it runs between structure lines). Without one: horizontal along the line to `inverter_x`.
  - Then the three intermediate points and the inverter point. The trailing space ends the PLINE command.
- `distance1 = 6.6553` (inverter to the structure line above it) and `distance2 = 2` (to the line below). Both are hardcoded.
- Patterns: `R_A_W` and `L_A_E` use `+distance1`, `R_B_W` and `L_B_E` use `−distance2`, and the other four go straight (pattern 3). All 8 variants are computed for every row, and `np.select` picks the one matching `Relative_Position`.
- **Warning:** in the trench/duct choices, `L_B_E` uses `+distance2`, while the cable `L_B_E` uses `-distance2`. For that case the trench and ducts go 2 *above* the inverter instead of below, which looks like a typo.
- **Script Cabos - Reask.scr:** `PLINEWID 0`. The same polylines are written twice, on `EMF_cabo_solar_positivo` and on `EMF_cabo_solar_negativo`, so the red and black cables overlap exactly. Also exports `cables.xlsx`.
- **Script Valas - Reask.scr:** `PLINEWID 0.5`, layer `EMF_DC_vala`, one polyline per SubGroup. Also exports `script.xlsx`.
- **Script Dutos - Reask.scr:** `PLINEWID 0.03`. It repeats the trench polyline once per duct, from the digits of `Duct Definition`: one line for a 1-string duct and one for a 2-string duct, and one line per 3-string duct. Each goes on `EMF_bt_eletroduto_1S`, `_2S` or `_3S`. The three `np.select` calls are identical.
- All output paths are hardcoded (`Desktop\Pessoal\Python\Ecotechne\Dutos DC` and the Desktop) and overwritten on every run. The DuckDB connection is closed at the end.

In [13]:
script_df = groups_df[['No. of Strings', 'Duct Definition', 'Number of Ducts', 'closest_x', 'Y', 'inverter_x', 'inverter_y', 'Relative_Position']].copy()

def intermediary_coordinates(df):
    df['intermediary_y_1'] = np.where(
        df['inverter_y'] > df['Y'],
        df['inverter_y'] - 1.7,
        df['inverter_y'] + 1.7
    )

    df['intermediary_y_2'] = np.where(
        df['inverter_y'] > df['Y'],
        df['inverter_y'] - 1.5,
        df['inverter_y'] + 1.5
    )

    df['intermediary_y_3'] = np.where(
        df['inverter_y'] > df['Y'],
        df['inverter_y'] - 1.3,
        df['inverter_y'] + 1.3
    )

    df['inverter_y'] = np.where(
        df['inverter_y'] > df['Y'],
        df['inverter_y'] - 0.1,
        df['inverter_y'] + 0.1
    )

intermediary_coordinates(cables_df)
intermediary_coordinates(script_df)

script_df.rename(columns={'Y': 'Line'}, inplace=True)

'''
RED AND BLACK CABLES
This script generates the script according to the position of the inverter in relation to the structure, it has three patterns:

R_A_W, L_A_E = pattern 1
R_B_W, L_B_E = pattern 2
L_A_W, L_B_W, R_A_E, R_B_E = pattern 3

Pattern 1 generates a polyline that has 8 points:
1. End of the string
2. Pile coordinates
3. Pile X coordinates and Line of the Structure directly above the inverter
4. Inverter X and Line of the Structure directly above the inverter
5 to 7. Intermediary coordinates for stretching purposes
8. Inverter coordinates

Pattern 2 generates a polyline that has 8 points:
1. End of the string
2. Pile coordinates
3. Pile X coordinates and Line of the Structure directly below the inverter
4. Inverter X and Line of the Structure directly below the inverter
5 to 7. Intermediary coordinates for stretching purposes
8. Inverter coordinates

Pattern 3 generates a polyline that has 7 points:
1. End of the string
2. Pile coordinates
3. Inverter X and String Y coordinates
4 to 6. Intermediary coordinates for stretching purposes
7. Inverter coordinates
'''

def make_pline(row, is_cable, offset=None):
    segments = []
    if is_cable:
        segments.extend([f"{row['Exit']},{row['Line']}"])

    segments.extend([f"{row['closest_x']},{row['Line']}"])

    if offset is not None:
        segments.extend([
            f"{row['closest_x']},{row['inverter_y'] + offset}",
            f"{row['inverter_x']},{row['inverter_y'] + offset}"
        ])
    else:
        segments.extend([f"{row['inverter_x']},{row['Line']}"])

    segments.extend([
        f"{row['inverter_x']},{row['intermediary_y_1']}",
        f"{row['inverter_x']},{row['intermediary_y_2']}",
        f"{row['inverter_x']},{row['intermediary_y_3']}",
        f"{row['inverter_x']},{row['inverter_y']} "
    ])

    return "_.pline " + " ".join(segments)

distance1 = 6.6553 # Distance between the inverter and the structure line above it
distance2 = 2 # Distance between the inverter and the structure line below it

cable_choice_R_A_W = cables_df.apply(lambda row: make_pline(row, True, distance1), axis=1)
cable_choice_R_B_W = cables_df.apply(lambda row: make_pline(row, True, -distance2), axis=1)
cable_choice_L_A_W = cables_df.apply(lambda row: make_pline(row, True), axis=1)
cable_choice_L_B_W = cables_df.apply(lambda row: make_pline(row, True), axis=1)
cable_choice_R_A_E = cables_df.apply(lambda row: make_pline(row, True), axis=1)
cable_choice_R_B_E = cables_df.apply(lambda row: make_pline(row, True), axis=1)
cable_choice_L_A_E = cables_df.apply(lambda row: make_pline(row, True, distance1), axis=1)
cable_choice_L_B_E = cables_df.apply(lambda row: make_pline(row, True, -distance2), axis=1)

trench_or_duct_choice_R_A_W = script_df.apply(lambda row: make_pline(row, False, distance1), axis=1)
trench_or_duct_choice_R_B_W = script_df.apply(lambda row: make_pline(row, False, -distance2), axis=1)
trench_or_duct_choice_L_A_W = script_df.apply(lambda row: make_pline(row, False), axis=1)
trench_or_duct_choice_L_B_W = script_df.apply(lambda row: make_pline(row, False), axis=1)
trench_or_duct_choice_R_A_E = script_df.apply(lambda row: make_pline(row, False), axis=1)
trench_or_duct_choice_R_B_E = script_df.apply(lambda row: make_pline(row, False), axis=1)
trench_or_duct_choice_L_A_E = script_df.apply(lambda row: make_pline(row, False, distance1), axis=1)
trench_or_duct_choice_L_B_E = script_df.apply(lambda row: make_pline(row, False, distance2), axis=1)


cable_conditions = [cables_df['Relative_Position'] == 'R_A_W',
             cables_df['Relative_Position'] == 'R_B_W',
             cables_df['Relative_Position'] == 'L_A_W',
             cables_df['Relative_Position'] == 'L_B_W',
             cables_df['Relative_Position'] == 'R_A_E',
             cables_df['Relative_Position'] == 'R_B_E',
             cables_df['Relative_Position'] == 'L_A_E',
             cables_df['Relative_Position'] == 'L_B_E']

cable_choices = [cable_choice_R_A_W,
                 cable_choice_R_B_W,
                 cable_choice_L_A_W,
                 cable_choice_L_B_W,
                 cable_choice_R_A_E,
                 cable_choice_R_B_E,
                 cable_choice_L_A_E,
                 cable_choice_L_B_E]

cables_df['Script Cabos'] = np.select(cable_conditions, cable_choices, default='unk')
cables_df.to_excel(r"C:\Users\Usuario\Desktop\cables.xlsx")

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\Script Cabos - Reask.scr", 'w') as f:
    f.write('PLINEWID 0\n')
    f.write("LAYER S EMF_cabo_solar_positivo\n\n")
    f.write('\n'.join(cables_df["Script Cabos"].astype(str)) + '\n')
    f.write("LAYER S EMF_cabo_solar_negativo\n\n")
    f.write('\n'.join(cables_df["Script Cabos"].astype(str)) + '\n')


'''
TRENCHES
This script generates the script according to the position of the inverter in relation to the structure, it has three patterns:

R_A_W, L_A_E = pattern 1
R_B_W, L_B_E = pattern 2
L_A_W, L_B_W, R_A_E, R_B_E = pattern 3

Pattern 1 generates a polyline that has 7 points:
1. Pile coordinates
2. Pile X coordinates and Line of the Structure directly above the inverter
3. Inverter X and Line of the Structure directly above the inverter
4 to 6. Intermediary coordinates for stretching purposes
7. Inverter coordinates

Pattern 2 generates a polyline that has 7 points:
1. Pile coordinates
2. Pile X coordinates and Line of the Structure directly below the inverter
3. Inverter X and Line of the Structure directly below the inverter
4 to 6. Intermediary coordinates for stretching purposes
7. Inverter coordinates

Pattern 3 generates a polyline that has 6 points:
1. Pile coordinates
2. Inverter X and String Y coordinates
3 to 5. Intermediary coordinates for stretching purposes
6. Inverter coordinates
'''

trench_or_duct_conditions = [script_df['Relative_Position'] == 'R_A_W',
             script_df['Relative_Position'] == 'R_B_W',
             script_df['Relative_Position'] == 'L_A_W',
             script_df['Relative_Position'] == 'L_B_W',
             script_df['Relative_Position'] == 'R_A_E',
             script_df['Relative_Position'] == 'R_B_E',
             script_df['Relative_Position'] == 'L_A_E',
             script_df['Relative_Position'] == 'L_B_E']

trench_choices = [trench_or_duct_choice_R_A_W,
                  trench_or_duct_choice_R_B_W,
                  trench_or_duct_choice_L_A_W,
                  trench_or_duct_choice_L_B_W,
                  trench_or_duct_choice_R_A_E,
                  trench_or_duct_choice_R_B_E,
                  trench_or_duct_choice_L_A_E,
                  trench_or_duct_choice_L_B_E]

script_df['Script Valas'] = np.select(trench_or_duct_conditions, trench_choices, default='unk')
script_df.to_excel(r"C:\Users\Usuario\Desktop\script.xlsx")


with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\Script Valas - Reask.scr", 'w') as f:
    f.write('PLINEWID 0.5\n')
    f.write("LAYER S EMF_DC_vala\n\n")
    f.write('\n'.join(script_df["Script Valas"].astype(str)) + '\n')


'''
DUCTS
This script generates the script according to the position of the inverter in relation to the structure, it has three patterns:

R_A_W, L_A_E = pattern 1
R_B_W, L_B_E = pattern 2
L_A_W, L_B_W, R_A_E, R_B_E = pattern 3

Pattern 1 generates a polyline that has 7 points:
1. Pile coordinates
2. Pile X coordinates and Line of the Structure directly above the inverter
3. Inverter X and Line of the Structure directly above the inverter
4 to 6. Intermediary coordinates for stretching purposes
7. Inverter coordinates

Pattern 2 generates a polyline that has 7 points:
1. Pile coordinates
2. Pile X coordinates and Line of the Structure directly below the inverter
3. Inverter X and Line of the Structure directly below the inverter
4 to 6. Intermediary coordinates for stretching purposes
7. Inverter coordinates

Pattern 3 generates a polyline that has 6 points:
1. Pile coordinates
2. Inverter X and String Y coordinates
3 to 5. Intermediary coordinates for stretching purposes
6. Inverter coordinates

For Script Dutos 3, it also has the multiplication to account for the number of ducts with 3 strings that have the same coordinates.
'''

duct_choices = [trench_or_duct_choice_R_A_W,
                  trench_or_duct_choice_R_B_W,
                  trench_or_duct_choice_L_A_W,
                  trench_or_duct_choice_L_B_W,
                  trench_or_duct_choice_R_A_E,
                  trench_or_duct_choice_R_B_E,
                  trench_or_duct_choice_L_A_E,
                  trench_or_duct_choice_L_B_E]


script_df['Script Dutos 1'] = np.select(trench_or_duct_conditions, duct_choices, default='unk')
script_df['Script Dutos 2'] = np.select(trench_or_duct_conditions, duct_choices, default='unk')
script_df['Script Dutos 3'] = np.select(trench_or_duct_conditions, duct_choices, default='unk')

script_df['Script Dutos 1'] = script_df.apply(
    lambda row: (row['Script Dutos 1']) * str(row['Duct Definition']).count('1'),
    axis=1
)
script_df['Script Dutos 2'] = script_df.apply(
    lambda row: (row['Script Dutos 2']) * str(row['Duct Definition']).count('2'),
    axis=1
)

script_df['Script Dutos 3'] = script_df.apply(
    lambda row: (row['Script Dutos 3'] + "\n") * str(row['Duct Definition']).count('3'),
    axis=1
)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\Script Dutos - Reask.scr", 'w') as f:
    f.write('PLINEWID 0.03\n')
    # Layer 1
    layer1_commands = script_df["Script Dutos 1"].astype(str)
    layer1_commands = layer1_commands[layer1_commands != '']  # Remove empty strings
    if len(layer1_commands) > 0:  # Only write layer if there are commands
        f.write("LAYER S EMF_bt_eletroduto_1S\n\n")
        f.write('\n'.join(layer1_commands) + '\n')

    # Layer 2
    layer2_commands = script_df["Script Dutos 2"].astype(str)
    layer2_commands = layer2_commands[layer2_commands != '']
    if len(layer2_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_2S\n\n")
        f.write('\n'.join(layer2_commands) + '\n')

    # Layer 3
    layer3_commands = script_df["Script Dutos 3"].astype(str)
    layer3_commands = layer3_commands[layer3_commands != '']
    if len(layer3_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_3S\n\n")
        f.write(''.join(layer3_commands) + '\n')


conn.close()